In [1]:
import sqlite3
import pandas as pd
import numpy as np

# 1. DB 연결
db_path = 'chembl_36/chembl_36_sqlite/chembl_36.db' 
conn = sqlite3.connect(db_path)

# =========================================================
# [Table 1] 타겟 메인 정보 (1 타겟 = 1행 지향)
# =========================================================
# 타겟의 기본 정보와 UniProt ID(Accession)를 가져옵니다.
print("1. 타겟 메인 정보(Main) 추출 중...")

query_target_main = """
    SELECT DISTINCT
        td.chembl_id AS target_chembl_id,
        td.pref_name AS target_name,
        td.target_type AS target_type,
        td.organism AS organism,
        cs.accession AS uniprot_id
    FROM
        TARGET_DICTIONARY td
    LEFT JOIN
        TARGET_COMPONENTS tc ON td.tid = tc.tid
    LEFT JOIN
        COMPONENT_SEQUENCES cs ON tc.component_id = cs.component_id
    WHERE
        td.chembl_id IS NOT NULL
"""
df_target_main = pd.read_sql_query(query_target_main, conn)

# =========================================================
# [Table 2] 타겟 이명 매핑 테이블 (1 타겟 = N행)
# =========================================================
# ChEMBL ID와 Synonym을 연결합니다.
# 유전자 심볼(Gene Symbol)인지 일반 이름인지 구분하기 위해 syn_type도 가져옵니다.
print("2. 타겟 이명(Synonym) 매핑 테이블 추출 중...")

query_target_synonyms = """
    SELECT
        td.chembl_id AS target_chembl_id,
        -- [데이터 세정] 따옴표(") 제거 및 공백 정리
        TRIM(REPLACE(syn.component_synonym, '"', '')) AS synonym,
        syn.syn_type AS synonym_type
    FROM
        TARGET_DICTIONARY td
    JOIN
        TARGET_COMPONENTS tc ON td.tid = tc.tid
    JOIN
        COMPONENT_SYNONYMS syn ON tc.component_id = syn.component_id
    WHERE
        td.chembl_id IS NOT NULL
        AND syn.component_synonym IS NOT NULL
"""
df_target_synonyms = pd.read_sql_query(query_target_synonyms, conn)

conn.close()

# =========================================================
# [데이터 전처리] NaN -> None 변환
# =========================================================
df_target_main = df_target_main.replace({np.nan: None})
df_target_synonyms = df_target_synonyms.replace({np.nan: None})

# =========================================================
# [결과 확인]
# =========================================================
print("\n" + "="*40)
print(f"[타겟 메인 테이블] 크기: {df_target_main.shape}")
print("="*40)
print(df_target_main.head())

print("\n" + "="*40)
print(f"[타겟 이명 테이블] 크기: {df_target_synonyms.shape}")
print("="*40)
print(df_target_synonyms.head())

# (확인용) 특정 타겟(예: CHEMBL203 - EGFR)의 이명만 뽑아보

# 저장 예시
# df_target_main.to_csv('chembl_target_main.csv', index=False)
# df_target_synonyms.to_csv('chembl_target_synonyms.csv', index=False)

1. 타겟 메인 정보(Main) 추출 중...
2. 타겟 이명(Synonym) 매핑 테이블 추출 중...

[타겟 메인 테이블] 크기: (21581, 5)
  target_chembl_id                                        target_name  \
0    CHEMBL1075021                               DNA gyrase subunit B   
1    CHEMBL1075022                                               ParE   
2    CHEMBL1075023  5'-methylthioadenosine/S-adenosylhomocysteine ...   
3    CHEMBL1075024             Enoyl-(Acyl-carrier-protein) reductase   
4    CHEMBL1075025                              Isoprenyl transferase   

      target_type                                           organism  \
0  SINGLE PROTEIN  Streptococcus pneumoniae serotype 4 (strain AT...   
1  SINGLE PROTEIN                           Streptococcus pneumoniae   
2  SINGLE PROTEIN  Streptococcus pneumoniae serotype 4 (strain AT...   
3  SINGLE PROTEIN                           Streptococcus pneumoniae   
4  SINGLE PROTEIN  Streptococcus pneumoniae serotype 4 (strain AT...   

   uniprot_id  
0      P0A4L9  
1      Q7

In [2]:
df_target_main

,target_chembl_id,target_name,target_type,organism,uniprot_id
0,CHEMBL1075021,DNA gyrase subunit B,SINGLE PROTEIN,Streptococcus pneumoniae serotype 4 (strain AT...,P0A4L9
1,CHEMBL1075022,ParE,SINGLE PROTEIN,Streptococcus pneumoniae,Q7X1P7
2,CHEMBL1075023,5'-methylthioadenosine/S-adenosylhomocysteine ...,SINGLE PROTEIN,Streptococcus pneumoniae serotype 4 (strain AT...,A0A0H2UPP7
3,CHEMBL1075024,Enoyl-(Acyl-carrier-protein) reductase,SINGLE PROTEIN,Streptococcus pneumoniae,Q9FBC5
4,CHEMBL1075025,Isoprenyl transferase,SINGLE PROTEIN,Streptococcus pneumoniae serotype 4 (strain AT...,Q97SR4
...,...,...,...,...,...
21576,CHEMBL6194,Pteridine reductase 1,SINGLE PROTEIN,Leishmania major,Q01782
21577,CHEMBL6195,Ubiquitin carboxyl-terminal hydrolase isozyme L3,SINGLE PROTEIN,Homo sapiens,P15374
21578,CHEMBL6196,UDP-glucuronosyltransferase 2B4,SINGLE PROTEIN,Homo sapiens,P06133
21579,CHEMBL6197,Transporter,SINGLE PROTEIN,Macaca mulatta,Q7JHP9


In [3]:
data_dir = "chembl-data/target"

In [4]:
df_target_main

,target_chembl_id,target_name,target_type,organism,uniprot_id
0,CHEMBL1075021,DNA gyrase subunit B,SINGLE PROTEIN,Streptococcus pneumoniae serotype 4 (strain AT...,P0A4L9
1,CHEMBL1075022,ParE,SINGLE PROTEIN,Streptococcus pneumoniae,Q7X1P7
2,CHEMBL1075023,5'-methylthioadenosine/S-adenosylhomocysteine ...,SINGLE PROTEIN,Streptococcus pneumoniae serotype 4 (strain AT...,A0A0H2UPP7
3,CHEMBL1075024,Enoyl-(Acyl-carrier-protein) reductase,SINGLE PROTEIN,Streptococcus pneumoniae,Q9FBC5
4,CHEMBL1075025,Isoprenyl transferase,SINGLE PROTEIN,Streptococcus pneumoniae serotype 4 (strain AT...,Q97SR4
...,...,...,...,...,...
21576,CHEMBL6194,Pteridine reductase 1,SINGLE PROTEIN,Leishmania major,Q01782
21577,CHEMBL6195,Ubiquitin carboxyl-terminal hydrolase isozyme L3,SINGLE PROTEIN,Homo sapiens,P15374
21578,CHEMBL6196,UDP-glucuronosyltransferase 2B4,SINGLE PROTEIN,Homo sapiens,P06133
21579,CHEMBL6197,Transporter,SINGLE PROTEIN,Macaca mulatta,Q7JHP9


In [5]:
df_target_main.to_csv(data_dir+"/targets.csv", index=False)

In [6]:
df_target_synonyms

,target_chembl_id,synonym,synonym_type
0,CHEMBL2074,3.2.1.20,EC_NUMBER
1,CHEMBL2074,"Alpha-1,4-glucosidase",UNIPROT
2,CHEMBL2074,MGA,GENE_SYMBOL_OTHER
3,CHEMBL2074,MGAM,GENE_SYMBOL
4,CHEMBL2074,MGAML,GENE_SYMBOL_OTHER
...,...,...,...
146350,CHEMBL6067629,Folate receptor 2,UNIPROT
146351,CHEMBL6067629,Folate receptor beta,UNIPROT
146352,CHEMBL6067629,Folate-binding protein 2,UNIPROT
146353,CHEMBL6067629,Folbp2,GENE_SYMBOL_OTHER


In [7]:
df_target_synonyms.to_csv(data_dir+"/target_synonyms.csv", index=False)

In [8]:
df_target_synonyms["synonym_type"].value_counts()

synonym_type
UNIPROT              69704
GENE_SYMBOL_OTHER    50999
GENE_SYMBOL          16128
EC_NUMBER             9524
Name: count, dtype: int64